## What is Endee?

<p align="center">
  <img src="https://endee.io/favicon.png" width="80">
</p>

**Endee** is a high-performance **vector database and semantic search engine** designed to store embeddings and perform fast similarity searches across large datasets.

In this project, we demonstrate a **Retrieval-Augmented Generation (RAG) pipeline** using Endee.

### Workflow

1. Extract text from a PDF document  
2. Split the text into overlapping chunks  
3. Generate vector embeddings for each chunk  
4. Store the embeddings in **Endee**  
5. Retrieve relevant chunks and generate answers using a **Groq LLM**

### Documentation

For more details about Endee and vector search:

 https://docs.endee.io/

### **Importing Required Libraries**

In this step, we import the necessary libraries required to build our semantic search pipeline.

- **fitz (PyMuPDF)** – Used to read PDF files and extract text from them.
- **Endee** – A vector database used to store and search text embeddings efficiently.
- **SentenceTransformer** – A pretrained model used to convert text into numerical vector embeddings.
- **ChatGroq** – Provides access to Groq's Large Language Models for generating responses.
- **ChatPromptTemplate** – Helps structure prompts that are sent to the language model.
- **StrOutputParser** – Converts the language model’s output into a clean text format.

These libraries together enable us to build a system that extracts text from PDFs, converts it into embeddings, stores it in a vector database, retrieves relevant information, and generates answers using an LLM.

In [3]:
import fitz
from endee import Endee, Precision

from sentence_transformers import SentenceTransformer

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

c:\Users\Dell\Desktop\Shivang_RAG_Endee\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Loading Environment Variables

In this step, we load environment variables from a `.env` file to securely access sensitive information such as API keys.

- **dotenv** is used to load variables stored in a `.env` file into the environment.
- The `load_dotenv()` function reads the `.env` file and makes its variables available to the program.
- We then retrieve the **GROQ API key** using `os.getenv("GROQ_API_KEY")`.

This approach helps keep sensitive credentials secure and prevents hardcoding API keys directly in the code.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

### Configuration Parameters

In this step, we define configuration parameters that control the behavior of the RAG pipeline, including the language model, text chunking strategy, and retrieval settings.

- **GROQ_MODEL** – Specifies the Groq LLM model used to generate responses.
- **INDEX_NAME** – The name of the vector index in Endee where embeddings will be stored.
- **CHUNK_SIZE** – The maximum number of characters in each text chunk created from the PDF.
- **CHUNK_OVERLAP** – The number of overlapping characters between consecutive chunks to preserve context.
- **TOP_K** – The number of most relevant chunks retrieved from the vector database during search.
- **TEMPERATURE** – Controls the randomness of the LLM output. Lower values make responses more deterministic.
- **PDF_PATH** – The file path of the PDF document from which text will be extracted.

In [ ]:
GROQ_MODEL  = "llama-3.1-8b-instant"
INDEX_NAME  = "pdf_rag_index"
CHUNK_SIZE  = 600
CHUNK_OVERLAP = 100
TOP_K       = 4
TEMPERATURE = 0.2

PDF_PATH = "Shivang_Resume.pdf"

### Step 1: Extract Text from the PDF

In this step, we extract the raw text from the PDF document. This text will later be split into smaller chunks and converted into vector embeddings for semantic search.

- The function **`extract_pdf_text()`** opens the PDF using **PyMuPDF (fitz)**.
- It iterates through each page of the document.
- Text from every page is extracted using `page.get_text()`.
- The extracted text from all pages is combined into a single string.
- The code also prints useful information such as:
  - Total number of pages in the PDF
  - Number of characters extracted from each page
  - A preview of the first 500 characters of the extracted text

This step ensures that all textual content from the PDF is collected before moving on to chunking and embedding.

In [ ]:
def extract_pdf_text(pdf_path: str) -> str:
    """Extract all text from a PDF file page by page."""
    doc = fitz.open(pdf_path)
    full_text = ""

    print(f"PDF has {len(doc)} page(s)")

    for page_num, page in enumerate(doc):
        page_text = page.get_text()
        full_text += page_text + "\n"
        print(f"Page {page_num + 1} → {len(page_text)} characters")

    doc.close()
    return full_text


print("Extracting text from PDF...\n")
raw_text = extract_pdf_text(PDF_PATH)

print(f"\nTotal characters extracted : {len(raw_text)}")
print(f"\nPREVIEW (first 500 chars)")
print(raw_text[:500])

### Preview Extracted Text

In this step, we perform a quick check on the extracted text.

- `len(raw_text)` prints the **total number of characters** extracted from the PDF.
- `raw_text[:500]` displays the **first 500 characters** of the extracted text.

This helps verify that the text extraction from the PDF was successful before proceeding to the chunking step.

In [ ]:

print(len(raw_text))
print(raw_text[:500])

### Step 2: Text Chunking

In this step, the extracted PDF text is divided into smaller **overlapping chunks**. Chunking is important because large documents cannot be directly processed by embedding models or LLMs.

- The function **`chunk_text()`** splits the text into smaller pieces based on a fixed size.
- The text is first split into **words** and then grouped into chunks whose length is controlled by **`CHUNK_SIZE`**.
- **Chunk overlap** is introduced using the **`CHUNK_OVERLAP`** parameter. This ensures that some words from the end of one chunk are repeated at the beginning of the next chunk.
- Overlapping chunks help preserve **context across chunk boundaries**, improving retrieval quality in RAG systems.

After creating the chunks:
- The code prints the **total number of chunks created**.
- It also displays **Chunk 0 and Chunk 1** as examples to verify the chunking process.

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list:
    """Split text into fixed-size word-boundary aligned chunks.

    The `overlap` parameter controls how many words from the end of one chunk
    are kept at the beginning of the next chunk, which helps preserve context
    across chunk boundaries for retrieval.
    """
    words = text.split()
    chunks = []

    i = 0
    while i < len(words):
        j = i
        curr_len = 0

        # Build a chunk until we reach the target size
        while j < len(words):
            add_len = len(words[j]) + (1 if j > i else 0)
            if curr_len + add_len > chunk_size:
                break
            curr_len += add_len
            j += 1

        chunks.append(" ".join(words[i:j]))

        if j >= len(words):
            break

        # Slide the window back by `overlap` words for the next chunk
        i = max(j - overlap, i + 1)

    return chunks


chunks = chunk_text(raw_text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)

print(f"Created {len(chunks)} chunks")
print(f"\n--- Chunk 0 ---\n{chunks[0]}")
print(f"\n--- Chunk 1 ---\n{chunks[1]}")

### Step 3: Generate Text Embeddings

In this step, we initialize a **sentence-transformer model** to convert text chunks into numerical vector representations called **embeddings**.

- The **`SentenceTransformer`** model `"all-MiniLM-L6-v2"` is loaded.  
- This model transforms text into **dense vectors** that capture the semantic meaning of the text.
- These embeddings allow us to perform **semantic similarity search** in the vector database.

To ensure the model loads correctly, a **quick sanity check** is performed:
- A simple sentence `"Hello world"` is encoded into a vector.
- If the model runs successfully, it confirms that the embedding model is ready for use.

These embeddings will later be stored in **Endee**, enabling efficient semantic retrieval.

In [ ]:

embedder = SentenceTransformer("all-MiniLM-L6-v2")


test_vec = embedder.encode(["Hello world"])[0]
print(f"   Vector dimensions : {len(test_vec)}")

### Step 4: Connect to the Vector Database (Endee)

In this step, we connect to the **Endee vector database** and ensure that the required index for storing embeddings exists.

- An **Endee client** is created to connect to the local Endee server running at `http://localhost:8080`.
- We attempt to **create a vector index** using the specified `INDEX_NAME`.
- The index is configured with:
  - **dimension = 384** → Matches the embedding size produced by the `all-MiniLM-L6-v2` model.
  - **space_type = "cosine"** → Uses cosine similarity to measure vector similarity.
  - **precision = INT8** → Reduces memory usage by storing vectors with lower precision.
- If the index already exists, the program simply **reuses the existing index** instead of creating a new one.
- Finally, the index is retrieved and prepared for storing embeddings.

This step prepares the **vector storage system** that will be used for semantic search in the RAG pipeline.

In [ ]:
endee_client = Endee()  
print(" Connected to Endee at localhost:8080")

try:
    endee_client.create_index(
        name=INDEX_NAME,
        dimension=384,         
        space_type="cosine",   
        precision=Precision.INT8
    )
    print(f"Index '{INDEX_NAME}' created!")

except Exception:
    print(f"Index '{INDEX_NAME}' already exists — reusing it.")

index = endee_client.get_index(name=INDEX_NAME)
print(f"Index ready: '{INDEX_NAME}'")

### Step 5: Store Embeddings in the Vector Database

In this step, the text chunks are converted into **vector embeddings** and stored in the Endee vector database for efficient semantic search.

- All text chunks are passed to the **embedding model** to generate vector representations.
- Each chunk is converted into a **numerical vector** that captures its semantic meaning.
- These vectors are then prepared as items containing:
  - **id** – A unique identifier for each chunk
  - **vector** – The embedding representation of the chunk
  - **meta** – Metadata storing the original text and chunk ID
- The vectors are inserted into the Endee index using the **`upsert()`** operation.

After this step, the PDF content is fully stored as **vector embeddings**, enabling fast and accurate **semantic search** across the document.

In [ ]:
print(f"Embedding {len(chunks)} chunks...\n")

# Embed all chunks in one batched call
vectors = embedder.encode(chunks, show_progress_bar=True)

print(f"\nUpserting into Endee...")

vector_items = [
    {
        "id": f"chunk_{i}",
        "vector": vec.tolist(),
        "meta": {"text": chunk, "chunk_id": i}
    }
    for i, (chunk, vec) in enumerate(zip(chunks, vectors))
]

index.upsert(vector_items)

print(f"\n{len(chunks)} chunks stored in Endee!")
print("   Your PDF is now a searchable vector index.")

### Initialize the Language Model

In this step, we initialize the **Groq Large Language Model (LLM)** that will be used to generate answers based on the retrieved document context.

- **ChatGroq** is used to interact with Groq’s hosted LLMs.
- The **`api_key`** authenticates the request using the GROQ API key loaded earlier.
- **`model`** specifies which Groq model will generate the responses.
- **`temperature`** controls the randomness of the output. Lower values make the responses more consistent and deterministic.

This LLM will later receive the **retrieved chunks from the vector database** along with the user's question and generate a final answer.

In [ ]:

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model=GROQ_MODEL,
    temperature=TEMPERATURE
)

### Create the RAG Prompt and Processing Chain

In this step, we define the **prompt template** and build the **RAG (Retrieval-Augmented Generation) chain** that will generate answers.

- A **ChatPromptTemplate** is created to structure how the question and retrieved context are sent to the LLM.
- The **system message** instructs the model to:
  - Answer only using the provided context.
  - Avoid making up information.
  - Respond concisely and clearly.
  - Return a specific message if the answer is not found in the context.
- The **human message** represents the user's question.

Next, a **RAG processing chain** is created:

- The **prompt template** formats the input.
- The **LLM (Groq model)** generates the response.
- The **StrOutputParser** converts the model output into clean text.

This chain allows us to pass the **retrieved document context and user question** to the model and receive a final answer.

In [ ]:

prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant that answers questions strictly based on the provided context.

Rules:
- Answer ONLY using the context below.
- If the answer is not in the context, say: 'I could not find relevant information in the document.'
- Be concise and clear.

Context:
{context}"""
    ),
    (
        "human",
        "{question}"
    )
])
rag_chain = prompt_template | llm | StrOutputParser()

### RAG Query Function

This function implements the **complete Retrieval-Augmented Generation (RAG) pipeline** using Endee, LangChain, and the Groq LLM. It processes a user question and generates an answer based on the information stored in the vector database.

The pipeline works in the following steps:

1. **Embed the Question**  
   - The user’s question is converted into a vector embedding using the **sentence-transformers model**.  
   - This allows the system to perform semantic similarity search.

2. **Retrieve Relevant Chunks**  
   - The question vector is sent to **Endee**, which searches the vector index.  
   - The **top-k most similar chunks** from the document are retrieved.

3. **Prepare Context**  
   - The retrieved chunks are combined into a single **context block**.  
   - This context will be provided to the language model.

4. **Generate the Answer**  
   - The context and the user’s question are passed into the **LangChain prompt template**.  
   - The prompt is then sent to the **Groq LLM**, which generates the final answer.  
   - The output is parsed into a clean text response.

This function essentially connects **retrieval (vector search)** with **generation (LLM)** to answer questions based strictly on the content of the document.

In [ ]:

def rag_query(question: str) -> str:
    """
    Full RAG pipeline using Endee + LangChain + Groq.

    Steps:
      1. Embed the question locally
      2. Search Endee for top-k similar chunks
      3. Inject chunks into LangChain prompt template
      4. Run chain → Groq LLM → plain string answer
    """
    print(f"\Question: {question}")
    print("-" * 55)

    # 1. Embed the question 
    print("[1/3] Embedding question with sentence-transformers...")
    query_vector = embedder.encode([question])[0].tolist()

    # 2. Search Endee 
    print(f"[2/3] Searching Endee (top {TOP_K} chunks)...")
    results = index.query(vector=query_vector, top_k=TOP_K)

    
    context = "\n\n".join([
        f"[Chunk {r['meta']['chunk_id']}]\n{r['meta']['text']}"
        for r in results
    ])

    print(f"\n--- Retrieved Context (preview) ---")
    print(context[:600])
    print("..." if len(context) > 600 else "")

    #3. Run LangChain chain
    print(f"\n   [3/3] Running LangChain → Groq ({GROQ_MODEL})...")
    answer = rag_chain.invoke({
        "context": context,
        "question": question
    })

    return answer

### Running a Sample Query

In this step, we test the RAG system by asking a question about the document.

- A sample question **"What is the main topic of this document?"** is defined.
- The question is passed to the **`rag_query()`** function.
- The function performs the complete RAG pipeline:
  - Embeds the question
  - Retrieves the most relevant chunks from Endee
  - Sends the context and question to the Groq LLM
- Finally, the generated answer is printed in a formatted output.

This step verifies that the **document has been successfully converted into a searchable knowledge base** and that the system can generate answers based on the document content.


In [ ]:

my_question = "What is the main topic of this document?"  

answer = rag_query(my_question)

print(f"\n{'=' * 60}")
print("FINAL ANSWER:")
print(f"{'=' * 60}")
print(answer)